Building off the first getting started notebook, now trying to refine everything into a bit of a nicer workflow. Goal is to read in all of the catchment and meterological data and combine them so that I can have just a csv of mean monthly values for each catchment for the last 30 years to make a climatology.

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import climate_indices as clindices
from tqdm import tqdm

First task is to read in all the catchment stuff so that I can store the met data by catchment

In [8]:
catchments = gpd.read_file(
    "DATA/Catchments/2025_FSU_Gauged_Catchment_Boundaries/2025 FSU Gauged Catchment Boundaries/2025 FSU Gauged Catchment Boundaries.shp"
)

# Read in and combine the catchment characteristics CSV from the FSU website with the geopandas catchment boundaries
# TODO: This might mess up the boundaries by converting gpd to pd so double check that. Nope, it doesn't seem to but leaving this here in case of bug later
catchment_descriptors = pd.read_csv("DATA/Catchments/2025+Flood+Studies+Update+(FSU)+Physical+Catchment+Descriptors+(gauged).csv")
catchments = catchments.merge(
    catchment_descriptors,
    on="LOCATION",
    how="left"
)

In [16]:
scarriff = catchments[catchments["LOCATION"] == 'SCARRIFF']
curranure = catchments[catchments["LOCATION"] == 'CURRANURE']
ballysadare = catchments[catchments["LOCATION"] == 'BALLYSADARE']
leixlip = catchments[catchments["LOCATION"] == 'LEIXLIP']

months = [
    "jan", "feb", "mar", "apr",
    "may", "jun", "jul", "aug",
    "sep", "oct", "nov", "dec"
]

Now to load in all of the gridded data for multiple years and have them in one df so that I can build the climatology

In [34]:
met_url = 'https://clidata.met.ie/cli/grids/latest/'

year_range_start = 1991
year_range_end = 2021

rainfall_gridded_data_sc = gpd.GeoDataFrame(columns=['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])
rainfall_gridded_data_cu = gpd.GeoDataFrame(columns=['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])
rainfall_gridded_data_ba = gpd.GeoDataFrame(columns=['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])
rainfall_gridded_data_le = gpd.GeoDataFrame(columns=['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'])

catchment_names = ['SCARRIFF', 'CURRANURE', 'BALLYSADARE', 'LEIXLIP']
catchment_results = {
    name: []
    for name in catchment_names
}

# TODO: There's a lot of reused code here to be turned into functions and I could definitely precompute some spatial stuff for efficiency
for year in tqdm(range(year_range_start, year_range_end)):
    rain_file_url = f'{met_url}rr_mon_grid_{year}.csv'
    mean_temp_url = f'{met_url}mntemp_mon_grid_{year}.csv'
    max_temp_url = f'{met_url}mmaxtemp_mon_grid_{year}.csv'
    min_temp_url = f'{met_url}mmintemp_mon_grid_{year}.csv'
    
    #print(rain_file_url)
    current_rainfall_year = pd.read_csv(rain_file_url)
    current_rainfall_year.columns = ['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    current_meantemp_year = pd.read_csv(mean_temp_url)
    current_meantemp_year.columns = ['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    current_maxtemp_year = pd.read_csv(max_temp_url)
    current_maxtemp_year.columns = ['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    current_mintemp_year = pd.read_csv(min_temp_url)
    current_mintemp_year.columns = ['east', 'north', 'jan', 'feb', 'mar', 'apr' ,'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

    # Convert the rainfall coords to geographic points
    rainfall_points = gpd.GeoDataFrame(
        current_rainfall_year,
        geometry=gpd.points_from_xy(
            current_rainfall_year["east"],
            current_rainfall_year["north"]
        ),
        crs=catchments.crs)
    meantemp_points = gpd.GeoDataFrame(
        current_meantemp_year,
        geometry=gpd.points_from_xy(
            current_meantemp_year["east"],
            current_meantemp_year["north"]
        ),
        crs=catchments.crs)
    maxtemp_points = gpd.GeoDataFrame(
        current_maxtemp_year,
        geometry=gpd.points_from_xy(
            current_maxtemp_year["east"],
            current_maxtemp_year["north"]
        ),
        crs=catchments.crs)
    mintemp_points = gpd.GeoDataFrame(
        current_mintemp_year,
        geometry=gpd.points_from_xy(
            current_mintemp_year["east"],
            current_mintemp_year["north"]
        ),
        crs=catchments.crs)

    for name in catchment_names:
        polygon = catchments[
            catchments["LOCATION"] == name
        ]

        rain_points = gpd.sjoin(
            rainfall_points,
            polygon,
            how="inner",
            predicate="within"
        )
        meantemp_points_in_catchment = gpd.sjoin(
            meantemp_points,
            polygon,
            how="inner",
            predicate="within"
        )
        maxtemp_points_in_catchment = gpd.sjoin(
            maxtemp_points,
            polygon,
            how="inner",
            predicate="within"
        )
        mintemp_points_in_catchment = gpd.sjoin(
            mintemp_points,
            polygon,
            how="inner",
            predicate="within"
        )

        rain_monthly_mean = rain_points[months].mean()
        meantemp_monthly_mean = meantemp_points_in_catchment[months].mean()
        maxtemp_monthly_mean = maxtemp_points_in_catchment[months].mean()
        mintemp_monthly_mean = mintemp_points_in_catchment[months].mean()

        for month_number, month in enumerate(
            months, start=1
        ):

            catchment_results[name].append({

                "Date": pd.Timestamp(
                    year=year,
                    month=month_number,
                    day=1
                ),

                "Rainfall": rain_monthly_mean[month],
                'MeanTemp': meantemp_monthly_mean[month],
                'MaxTemp': maxtemp_monthly_mean[month],
                'MinTemp': mintemp_monthly_mean[month]

            })
            
    

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [05:08<00:00, 10.29s/it]


In [35]:
for name, records in catchment_results.items():

    df = pd.DataFrame(records)

    df = df.sort_values("Date")

    df.to_csv(
        f"DATA/CatchmentTimeSeries/{name}_allvars.csv",
        index=False
    )

So all the data SHOULD be outputted to a file now! This notebook can then be tucked away and doesn't need to be used again (but kept to show how I made the csv) while I read in just the relevant data elsewhere woooooo! Plan now to to read it in and plot it quickly to make sure it all makes sense, then I can hopefully calculate the indices really really easily with this legwork done.